# Task 1 : Model Training
## Purchase Prediction: `is_reorder` Binary Classification

This notebook trains and evaluates a model to predict whether a given order is a reorder (`is_reorder = 1`) or a first-time purchase (`is_reorder = 0`).

## 1. Algorithm Shortlist and Comparison

The prediction task is a **binary classification** problem: given features about a customer, product, producer, quantity, price, and time context, predict whether the order is a reorder. Below we evaluate five candidate algorithms against the characteristics of this dataset.

---

### Candidate 1 : Random Forest

**What it is:** An ensemble of decision trees trained on random feature subsets. Each tree votes and the majority class wins.

**Why it suits this problem:**
- The dataset mixes categorical (customer type, product, season) and numerical (quantity, unit price, month) features. Random Forest handles both naturally after label encoding.
- With only 2,500 rows, training is fast and ensemble averaging controls overfitting.
- Feature importances are straightforward to extract, aiding interpretability in the report.
- `class_weight='balanced'` handles the ~93% majority class without resampling.

**Limitations:** Treats each row independently; sequential purchasing history and customer–product interaction patterns are lost unless features are manually engineered.

**Verdict:** Strong baseline. Interpretable, robust, and well-suited to tabular data of this size.

---

### Candidate 2 : XGBoost (Gradient Boosted Trees)

**What it is:** A sequential ensemble that adds trees one at a time, each correcting the residual errors of the previous using gradient descent in function space.

**Why it suits this problem:**
- Consistently outperforms Random Forest on structured/tabular datasets; boosting focuses learning on the hardest-to-classify examples.
- `scale_pos_weight` handles the class imbalance ratio (~13:1) directly without resampling.
- Built-in L1/L2 regularisation reduces overfitting on small datasets.
- Learns complex non-linear boundaries across features like customer type, product, and season.

**Limitations:** More hyperparameters to tune than Random Forest. Like Random Forest, treats rows independently with no built-in temporal awareness.

**Verdict:** Strong primary candidate. Typically the best-performing model on tabular classification tasks.

---

### Candidate 3 : LSTM (Long Short-Term Memory)

**What it is:** A recurrent neural network that learns long-range dependencies in sequential data by passing a hidden state forward across time steps.

**Why it could suit this problem:**
- Purchase prediction is inherently sequential; an LSTM could learn that a customer who bought Milk last week is likely to reorder it this week.
- Can capture temporal rhythms (weekly patterns, seasonal escalations) that tree-based methods miss without manual feature engineering.

**Why it is less suitable here:**
- The dataset has only 15 customers, giving per-customer sequences of ~167 orders : far too small a corpus for a neural sequence model to generalise reliably.
- The `is_reorder` label follows a simple rule (whether the customer–product pair appeared before), well-captured by a single aggregated count feature. A complex sequence model adds overhead without meaningful accuracy gain.
- Requires significant preprocessing (sequence construction, padding, batching) and is harder to interpret.
- No GPU is assumed in this environment; training is slower for marginal or no performance benefit.

**Verdict:** Theoretically interesting but impractical for this dataset's scale. More appropriate with tens of thousands of customers and a richer event history.

---

### Candidate 4 (Reference) : Collaborative Filtering

Collaborative filtering (matrix factorisation, user–item embeddings) is designed to answer *which* product a customer might buy next. Here the target is binary : reorder or not : for a given transaction row. This is a supervised classification framing, not a recommendation framing, so collaborative filtering does not directly apply. It could contribute latent customer–product affinity features as classifier inputs, but that is beyond the scope of this task.

---

### Candidate 5 (Reference) : ARIMA

ARIMA is a univariate time-series forecasting model for predicting future values of a single numeric variable (e.g. total weekly sales). It does not produce per-row binary predictions across multiple entities, so it is not applicable as a row-level classifier. It would be relevant if the goal were to forecast aggregate demand over time rather than classifying individual orders.

---

## Chosen Approach: XGBoost with Random Forest as Baseline

**XGBoost is selected as the primary model** for the following reasons:

1. **Task fit:** Binary classification on structured tabular data is the core use case for gradient boosted trees.
2. **Class imbalance:** `scale_pos_weight` handles the 93%/7% split directly without resampling.
3. **Feature types:** Mixed categorical and numerical features are well-handled by tree splits after label encoding.
4. **Interpretability:** SHAP values can be computed post-hoc to explain which features drive predictions.
5. **Practical constraints:** Small dataset, no GPU required, fast training, straightforward cross-validation tuning.

Random Forest is trained alongside as a **comparison baseline** to verify that XGBoost gains are real. LSTM and ARIMA are excluded due to data scale constraints and task framing mismatch respectively.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb

ROOT_DIR = Path.cwd().resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from task1_purchase_prediction.src.model import compare_models

DATA_DIR = Path('../data/raw')
orders_df = pd.read_csv(DATA_DIR / 'orders.csv')
print('Shape:', orders_df.shape)
orders_df.head()

Shape: (2500, 12)


,order_id,customer_id,customer_type,producer_id,product,quantity,unit_price,total_price,order_date,month,season,is_reorder
0,ORD001028,CUST006,restaurant,PROD004,Apples,16,2.06,32.96,2023-01-01,1,winter,1
1,ORD002254,CUST004,household,PROD006,Potatoes,6,1.32,7.92,2023-01-01,1,winter,1
2,ORD001712,CUST014,restaurant,PROD001,Potatoes,20,1.30,26.00,2023-01-02,1,winter,1
3,ORD001569,CUST001,household,PROD003,Croissant,3,1.08,3.24,2023-01-02,1,winter,1
4,ORD000542,CUST012,hotel,PROD008,Strawberries,6,3.38,20.28,2023-01-02,1,winter,1


## 2. Feature Engineering

In [3]:
df = orders_df.copy()

categorical_cols = ['customer_id', 'customer_type', 'producer_id', 'product', 'season']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

df = df.drop(columns=['order_id', 'order_date'])
X = df.drop(columns=['is_reorder'])
y = df['is_reorder']
print('Features:', list(X.columns))
print('Class distribution:')
print(y.value_counts())

Features: ['customer_id', 'customer_type', 'producer_id', 'product', 'quantity', 'unit_price', 'total_price', 'month', 'season']
Class distribution:
is_reorder
1    2323
0     177
Name: count, dtype: int64


## 3. Train/Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (2000, 9), Test: (500, 9)


In [5]:
comparison_results = compare_models(X_train, X_test, y_train, y_test)

rows = []
for model_name, metrics in comparison_results.items():
    if metrics.get('status') == 'ok':
        report = metrics['classification_report']
        rows.append(
            {
                'model': model_name,
                'accuracy': metrics['accuracy'],
                'macro_f1': report['macro avg']['f1-score'],
                'weighted_f1': report['weighted avg']['f1-score'],
            }
        )
    else:
        rows.append(
            {
                'model': model_name,
                'accuracy': None,
                'macro_f1': None,
                'weighted_f1': None,
                'status': metrics.get('reason', 'skipped'),
            }
        )

results_table = pd.DataFrame(rows).sort_values(by='accuracy', ascending=False, na_position='last')
print('=== Model Comparison Results ===')
display(results_table)

=== Model Comparison Results ===


,model,accuracy,macro_f1,weighted_f1
2,logistic_regression,0.930,0.481865,0.896269
0,random_forest,0.928,0.481328,0.895270
1,xgboost,0.908,0.475891,0.885157


## 4. Random Forest Baseline

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_rf))

## 5. XGBoost (Primary Model)

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f'scale_pos_weight = {scale_pos_weight:.2f}')

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42,
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print('=== XGBoost ===')
print(classification_report(y_test, y_pred_xgb))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred_xgb))

## 6. Save Models

In [ ]:
import joblib

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(rf, MODELS_DIR / 'random_forest.pkl')
xgb_model.save_model(str(MODELS_DIR / 'xgboost.json'))
joblib.dump(label_encoders, MODELS_DIR / 'label_encoders.pkl')
print('Models saved to', MODELS_DIR.resolve())